# 05 - Demonstração (vídeo)
Assistente do Calçada Livre: **RAG + limiar de 'não consta'** com o Qwen2.5-1.5B. Mostra (1) pergunta que consta, (2) pergunta que não consta e (3) explicação de uma ocorrência. Colab: Ambiente de execução > Alterar tipo > GPU T4.

In [1]:
# Setup
import os, sys, subprocess
REPO = "https://github.com/melo-zz/calcada-livre-genai.git"
if os.path.exists("/content"):
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())
print("pasta de trabalho:", os.getcwd())

pasta de trabalho: /content/repo


In [2]:
%pip install -q -r requirements.txt
%pip uninstall -y -q torchao

In [3]:
import json, os
from src.kb import carregar_trechos
from src.retrieval import Retriever
from src.llm import HFGen
from src.pipeline import Assistente
from avaliacao.casos import carregar
from src.retrieval import escolher_limiar
os.makedirs(".cache", exist_ok=True)
trechos = carregar_trechos("base/trechos.jsonl")
r = Retriever(trechos, cache=".cache/emb.npy")
limiar = escolher_limiar(carregar(), r)["limiar"]      # mesmo limiar do notebook 02
gen = HFGen()                                           # modelo base (configuração B: RAG simples)
assistente = Assistente(r, lambda m: gen.gerar(m, usar_adapter=False), limiar)
print("pronto | limiar:", round(limiar, 3), "| trechos:", len(trechos))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (16 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

pronto | limiar: 0.863 | trechos: 1258


In [4]:
def mostrar(pergunta):
    s = assistente.perguntar(pergunta)
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", s["resposta"])
    print("FONTES  :", s["fontes"])
    print("CONSTA  :", s["consta"], "| caminho:", s["caminho"], "| encaminhar para humano:", s["encaminhar_humano"])
    for c in s["contextos"]:
        print(f"   trecho {c['id']}  score={c['score']}")
    print()

## 1. Pergunta que CONSTA na base

In [17]:
mostrar("Os semáforos para pedestres precisam emitir sinal sonoro?")

PERGUNTA: Os semáforos para pedestres precisam emitir sinal sonoro?
RESPOSTA: Sim, os semáforos para pedestres podem emitir sinal sonoro suave, intermitente e sem estridência, conforme previsto na Lei 10.098/2000, Art. 9º.
FONTES  : ['l10098_art9']
CONSTA  : True | caminho: llm | encaminhar para humano: False
   trecho l10098_art9  score=0.9193
   trecho l13146_art112_p7  score=0.9083
   trecho ctb_art70  score=0.8871



## 2. Pergunta que NÃO CONSTA na base (o assistente recusa e encaminha para um humano)

In [19]:
mostrar("Qual o prazo para a prefeitura consertar um buraco na calçada?")

PERGUNTA: Qual o prazo para a prefeitura consertar um buraco na calçada?
RESPOSTA: Não encontrei essa informação na base consultada.
FONTES  : []
CONSTA  : False | caminho: abaixo_do_limiar | encaminhar para humano: True
   trecho ctb_art68_p2  score=0.8549
   trecho ctb_art94  score=0.8461
   trecho ctb_art246  score=0.8454



## 3. Explicação de uma ocorrência (detectada pelo carrinho, pela foto ou pelo app)

In [20]:
def explicar_ocorrencia(tipo):
    descricoes = {"buraco": "um buraco na calçada", "obstaculo": "um obstáculo no passeio", "piso_quebrado": "piso quebrado ou irregular",
                  "degrau_sem_rampa": "um degrau sem rampa", "trecho_escuro": "um trecho sem iluminação"}
    desc = descricoes.get(tipo, "uma barreira de acessibilidade")
    formulacoes = [
        f"Uma pessoa encontrou {desc} em uma calçada. Quais direitos e normas de acessibilidade se aplicam?",
        f"As vias públicas e os passeios precisam ser acessíveis? Quais normas se aplicam a {desc}?",
        "O que a lei diz sobre a acessibilidade de vias e espaços públicos?",
        "As vias públicas e os parques precisam ser projetados de forma acessível?",
    ]
    for f in formulacoes:
        s = assistente.perguntar(f)
        if s["consta"]:
            break
    print("OCORRÊNCIA:", tipo, "->", desc)
    print("PERGUNTA USADA:", f)
    print("RESPOSTA:", s["resposta"])
    print("FONTES  :", s["fontes"])
    print("CONSTA  :", s["consta"], "| encaminhar para humano:", s["encaminhar_humano"])
    for c in s["contextos"]:
        print(f"   trecho {c['id']}  score={c['score']}")

explicar_ocorrencia("degrau_sem_rampa")

OCORRÊNCIA: degrau_sem_rampa -> um degrau sem rampa
PERGUNTA USADA: As vias públicas e os passeios precisam ser acessíveis? Quais normas se aplicam a um degrau sem rampa?
RESPOSTA: Sim, as vias públicas e os passeios precisam ser acessíveis conforme determina a legislação brasileira.
FONTES  : ['l13146_art113_p3', 'd5296_art15_p1']
CONSTA  : True | encaminhar para humano: False
   trecho l13146_art113_p3  score=0.904
   trecho d5296_art15_p1  score=0.9022
   trecho l10098_art5  score=0.8981
